# Part 2 EDA - KTX 공실률 시간·계절 패턴 분석

`data/processed/ktx_long.csv`를 기준으로 노선별 공실률 현황, 극단 구간, 계절·월·연도별 패턴을 확인한다.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA_PATH = Path("../data/processed/ktx_long.csv")
OUTPUT_DIR = Path("../data/processed/eda_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATUS_COLOR_MAP = {
    "초과수요": "#d62728",
    "여유없음": "#ff7f0e",
    "공실있음": "#1f77b4",
}
SEASON_ORDER = ["봄", "여름", "가을", "겨울"]
ROUTE_ORDER = ["경부선", "호남선", "경전선", "전라선", "동해선"]

df = pd.read_csv(DATA_PATH, parse_dates=["yearmonth"])
df["개통전"] = (
    (df["노선"] == "경전선")
    & (df["yearmonth"] >= "2010-01-01")
    & (df["yearmonth"] <= "2010-11-01")
).astype(int)
df["계절"] = pd.Categorical(df["계절"], categories=SEASON_ORDER, ordered=True)
df["노선"] = pd.Categorical(df["노선"], categories=ROUTE_ORDER, ordered=True)
df["공실상태"] = pd.Categorical(
    df["공실상태"],
    categories=["초과수요", "여유없음", "공실있음"],
    ordered=True,
)

print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())

shape: (954, 16)
columns:
['yearmonth', '연도', '월', '분기', '계절', '코로나기간', 'SRT개통후', '노선', '여객수_천명', 'KTX이용률', '공실률', '초과수요', '공실상태', 'KTX전체이용률', 'KTX전체공실률', '개통전']


## 1. 기본 현황

In [2]:
route_status = (
    df.groupby("노선", observed=True)
    .agg(
        시작일=("yearmonth", "min"),
        종료일=("yearmonth", "max"),
        행수=("yearmonth", "size"),
    )
    .reset_index()
)
route_status["시작일"] = route_status["시작일"].dt.strftime("%Y-%m-%d")
route_status["종료일"] = route_status["종료일"].dt.strftime("%Y-%m-%d")
print("노선별 데이터 기간 및 행 수")
print(route_status.to_string(index=False))

print("\n공실률 기초통계")
print(df["공실률"].describe().round(2).to_string())

노선별 데이터 기간 및 행 수
 노선        시작일        종료일  행수
경부선 2004-04-01 2024-12-01 249
호남선 2004-04-01 2024-12-01 249
경전선 2010-01-01 2024-12-01 180
전라선 2011-10-01 2024-12-01 159
동해선 2015-04-01 2024-12-01 117

공실률 기초통계
count    954.00
mean      13.18
std       23.99
min      -55.00
25%       -4.00
50%        9.00
75%       28.00
max      100.00


## 2. 경전선 공실률 100% 구간

In [3]:
gyeongjeon_full_vacancy = df[(df["노선"] == "경전선") & (df["공실률"] == 100)]
print("경전선 공실률 100% 구간")
if gyeongjeon_full_vacancy.empty:
    print("없음")
else:
    print(
        gyeongjeon_full_vacancy[
            ["yearmonth", "노선", "KTX이용률", "공실률", "공실상태"]
        ].to_string(index=False)
    )

경전선 공실률 100% 구간
 yearmonth  노선  KTX이용률   공실률 공실상태
2010-01-01 경전선     0.0 100.0 공실있음
2010-02-01 경전선     0.0 100.0 공실있음
2010-03-01 경전선     0.0 100.0 공실있음
2010-04-01 경전선     0.0 100.0 공실있음
2010-05-01 경전선     0.0 100.0 공실있음
2010-06-01 경전선     0.0 100.0 공실있음
2010-07-01 경전선     0.0 100.0 공실있음
2010-08-01 경전선     0.0 100.0 공실있음
2010-09-01 경전선     0.0 100.0 공실있음
2010-10-01 경전선     0.0 100.0 공실있음
2010-11-01 경전선     0.0 100.0 공실있음


## 3. 동해선 공실률 -55% 구간

In [4]:
donghae_overdemand = df[(df["노선"] == "동해선") & (df["공실률"] == -55)]
print("동해선 공실률 -55% 구간")
if donghae_overdemand.empty:
    print("없음")
else:
    print(
        donghae_overdemand[
            ["yearmonth", "노선", "KTX이용률", "공실률", "공실상태"]
        ].to_string(index=False)
    )

동해선 공실률 -55% 구간
 yearmonth  노선  KTX이용률   공실률 공실상태
2018-05-01 동해선   155.0 -55.0 초과수요
2018-11-01 동해선   155.0 -55.0 초과수요


## 4. 계절별 공실률 Boxplot

In [5]:
fig_season = px.box(
    df,
    x="계절",
    y="공실률",
    color="공실상태",
    facet_col="노선",
    facet_col_wrap=3,
    category_orders={
        "계절": SEASON_ORDER,
        "노선": ROUTE_ORDER,
        "공실상태": ["초과수요", "여유없음", "공실있음"],
    },
    color_discrete_map=STATUS_COLOR_MAP,
    points="outliers",
    title="노선별 계절 공실률 분포",
)
fig_season.update_layout(width=1400, height=850, legend_title_text="공실상태")
fig_season.update_yaxes(title_text="공실률(%)")
fig_season.write_image(OUTPUT_DIR / "boxplot_season.png", scale=2)
print(OUTPUT_DIR / "boxplot_season.png")
fig_season.show()

/tmp/ipykernel_3804409/2589128793.py:19: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_season.write_image(OUTPUT_DIR / "boxplot_season.png", scale=2)


../data/processed/eda_results/boxplot_season.png


## 5. 월별 평균 공실률 Line Chart

In [6]:
monthly_avg = (
    df.groupby(["월", "노선"], observed=True)["공실률"]
    .mean()
    .reset_index()
)
fig_monthly = px.line(
    monthly_avg,
    x="월",
    y="공실률",
    color="노선",
    markers=True,
    category_orders={"노선": ROUTE_ORDER},
    title="월별 평균 공실률",
)
fig_monthly.update_layout(
    width=1200,
    height=650,
    xaxis=dict(dtick=1),
    legend_title_text="노선",
)
fig_monthly.update_yaxes(title_text="평균 공실률(%)")
fig_monthly.write_image(OUTPUT_DIR / "lineplot_monthly.png", scale=2)
print(OUTPUT_DIR / "lineplot_monthly.png")
fig_monthly.show()

/tmp/ipykernel_3804409/313790227.py:22: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_monthly.write_image(OUTPUT_DIR / "lineplot_monthly.png", scale=2)


../data/processed/eda_results/lineplot_monthly.png


## 6. 연도별 트렌드 Line Chart

In [7]:
yearly_avg = (
    df.groupby(["연도", "노선"], observed=True)["공실률"]
    .mean()
    .reset_index()
)
fig_yearly = px.line(
    yearly_avg,
    x="연도",
    y="공실률",
    color="노선",
    markers=True,
    category_orders={"노선": ROUTE_ORDER},
    title="연도별 평균 공실률 트렌드",
)
fig_yearly.add_vrect(
    x0=2020,
    x1=2022.5,
    fillcolor="gray",
    opacity=0.18,
    line_width=0,
    annotation_text="코로나 기간",
    annotation_position="top left",
)
fig_yearly.add_vline(
    x=2016.92,
    line_dash="dash",
    line_color="black",
    annotation_text="SRT 개통(2016-12)",
    annotation_position="top right",
)
fig_yearly.update_layout(
    width=1200,
    height=650,
    xaxis=dict(dtick=1),
    legend_title_text="노선",
)
fig_yearly.update_yaxes(title_text="연평균 공실률(%)")
fig_yearly.write_image(OUTPUT_DIR / "trend_yearly.png", scale=2)
print(OUTPUT_DIR / "trend_yearly.png")
fig_yearly.show()

../data/processed/eda_results/trend_yearly.png


/tmp/ipykernel_3804409/752775214.py:38: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_yearly.write_image(OUTPUT_DIR / "trend_yearly.png", scale=2)


## 7. 통계 검정

계절별 공실률 차이는 노선별 Kruskal-Wallis 검정으로 확인하고, 경부선과 호남선의 공실률 차이는 Mann-Whitney U test로 비교한다.

In [8]:
from scipy.stats import kruskal, mannwhitneyu

test_results = []

print('Kruskal-Wallis 검정')
print('귀무가설: 계절에 따라 공실률 차이가 없다')
for route in ROUTE_ORDER:
    route_df = df[df['노선'] == route]
    groups = [
        route_df.loc[route_df['계절'] == season, '공실률'].dropna()
        for season in SEASON_ORDER
    ]
    valid_groups = [group for group in groups if len(group) > 0]

    if len(valid_groups) < 2:
        p_value = float('nan')
        interpretation = '검정 불가'
    else:
        _, p_value = kruskal(*valid_groups)
        interpretation = (
            '계절별 유의미한 차이 있음'
            if p_value < 0.05
            else '유의미한 차이 없음'
        )

    test_results.append({
        '노선': route,
        '검정방법': 'Kruskal-Wallis',
        'p-value': p_value,
        '해석': interpretation,
    })
    print(f'{route}: p-value={p_value:.6f} → {interpretation}')

print('\nMann-Whitney U test')
print('비교: 경부선 vs 호남선 공실률')
gyeongbu = df.loc[df['노선'] == '경부선', '공실률'].dropna()
honam = df.loc[df['노선'] == '호남선', '공실률'].dropna()
_, p_value = mannwhitneyu(gyeongbu, honam, alternative='two-sided')
interpretation = (
    '경부선과 호남선 공실률 분포에 유의미한 차이 있음'
    if p_value < 0.05
    else '유의미한 차이 없음'
)
test_results.append({
    '노선': '경부선 vs 호남선',
    '검정방법': 'Mann-Whitney U',
    'p-value': p_value,
    '해석': interpretation,
})
print(f'경부선 vs 호남선: p-value={p_value:.6f} → {interpretation}')

print('\n개통전 플래그 확인')
print(df['개통전'].value_counts().sort_index().to_string())
print(df[df['개통전'] == 1][['yearmonth', '노선', '공실률', '개통전']].to_string(index=False))

test_summary = pd.DataFrame(test_results)
test_summary['p-value'] = test_summary['p-value'].round(6)
print('\n검정 결과 요약')
print(test_summary.to_string(index=False))
test_summary

Kruskal-Wallis 검정
귀무가설: 계절에 따라 공실률 차이가 없다
경부선: p-value=0.184222 → 유의미한 차이 없음
호남선: p-value=0.099781 → 유의미한 차이 없음
경전선: p-value=0.126483 → 유의미한 차이 없음
전라선: p-value=0.201151 → 유의미한 차이 없음
동해선: p-value=0.750775 → 유의미한 차이 없음

Mann-Whitney U test
비교: 경부선 vs 호남선 공실률
경부선 vs 호남선: p-value=0.000000 → 경부선과 호남선 공실률 분포에 유의미한 차이 있음

개통전 플래그 확인
개통전
0    943
1     11
 yearmonth  노선   공실률  개통전
2010-01-01 경전선 100.0    1
2010-02-01 경전선 100.0    1
2010-03-01 경전선 100.0    1
2010-04-01 경전선 100.0    1
2010-05-01 경전선 100.0    1
2010-06-01 경전선 100.0    1
2010-07-01 경전선 100.0    1
2010-08-01 경전선 100.0    1
2010-09-01 경전선 100.0    1
2010-10-01 경전선 100.0    1
2010-11-01 경전선 100.0    1

검정 결과 요약
        노선           검정방법  p-value                          해석
       경부선 Kruskal-Wallis 0.184222                  유의미한 차이 없음
       호남선 Kruskal-Wallis 0.099781                  유의미한 차이 없음
       경전선 Kruskal-Wallis 0.126483                  유의미한 차이 없음
       전라선 Kruskal-Wallis 0.201151                  유의미한 차이 없음
       동해선 Kr

,노선,검정방법,p-value,해석
0,경부선,Kruskal-Wallis,0.184222,유의미한 차이 없음
1,호남선,Kruskal-Wallis,0.099781,유의미한 차이 없음
2,경전선,Kruskal-Wallis,0.126483,유의미한 차이 없음
3,전라선,Kruskal-Wallis,0.201151,유의미한 차이 없음
4,동해선,Kruskal-Wallis,0.750775,유의미한 차이 없음
5,경부선 vs 호남선,Mann-Whitney U,0.000000,경부선과 호남선 공실률 분포에 유의미한 차이 있음


## 8. 연도 트렌드 제거 후 계절 효과 분석

노선별·연도별 평균 공실률을 제거한 `공실률_잔차`를 사용해 장기 연도 트렌드의 영향을 줄이고 계절 효과를 다시 확인한다.

잔차 해석:
- `공실률_잔차 > 0`: 같은 노선의 같은 연도 평균보다 공실률이 높음
- `공실률_잔차 < 0`: 같은 노선의 같은 연도 평균보다 공실률이 낮음

In [9]:
# 연도별 노선 평균 계산 후 잔차 생성
yearly_mean = df.groupby(['노선', '연도'], observed=True)['공실률'].transform('mean')
df['공실률_잔차'] = df['공실률'] - yearly_mean
df_detrended = df.dropna(subset=['공실률_잔차']).copy()

print(f"공실률_잔차 결측치 수: {df['공실률_잔차'].isna().sum()}")

fig_season_detrended = px.box(
    df_detrended,
    x='계절',
    y='공실률_잔차',
    color='공실상태',
    facet_col='노선',
    facet_col_wrap=3,
    category_orders={
        '계절': SEASON_ORDER,
        '노선': ROUTE_ORDER,
        '공실상태': ['초과수요', '여유없음', '공실있음'],
    },
    color_discrete_map=STATUS_COLOR_MAP,
    points='outliers',
    title='노선별 계절 공실률 잔차 분포',
)
fig_season_detrended.update_layout(
    width=1400,
    height=850,
    legend_title_text='공실상태',
)
fig_season_detrended.update_yaxes(title_text='공실률 잔차(%p)')
fig_season_detrended.add_hline(y=0, line_dash='dash', line_color='black')
fig_season_detrended.write_image(
    OUTPUT_DIR / 'boxplot_season_detrended.png',
    scale=2,
)
print(OUTPUT_DIR / 'boxplot_season_detrended.png')
fig_season_detrended.show()

공실률_잔차 결측치 수: 0


../data/processed/eda_results/boxplot_season_detrended.png


/tmp/ipykernel_3804409/3364666366.py:31: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_season_detrended.write_image(


### 잔차 기준 Kruskal-Wallis 재검정

In [10]:
SIGNIFICANCE_LEVEL = 0.05
detrended_test_results = []

print('잔차 기준 Kruskal-Wallis 검정')
print('귀무가설: 트렌드 제거 후 계절별 차이 없다')

for route in ROUTE_ORDER:
    route_df = df_detrended[df_detrended['노선'] == route]
    groups = [
        route_df.loc[route_df['계절'] == season, '공실률_잔차'].dropna()
        for season in SEASON_ORDER
    ]
    valid_groups = [group for group in groups if len(group) > 0]

    if len(valid_groups) < 2:
        p_value = float('nan')
        interpretation = '검정 불가'
    else:
        _, p_value = kruskal(*valid_groups)
        interpretation = (
            '트렌드 제거 후 계절별 유의미한 차이 있음'
            if p_value < SIGNIFICANCE_LEVEL
            else '트렌드 제거 후 유의미한 차이 없음'
        )

    detrended_test_results.append({
        '노선': route,
        '검정방법': 'Kruskal-Wallis (잔차)',
        'p-value': p_value,
        '해석': interpretation,
    })
    print(f'{route}: p-value={p_value:.6f} → {interpretation}')

detrended_test_summary = pd.DataFrame(detrended_test_results)
detrended_test_summary['p-value'] = detrended_test_summary['p-value'].round(6)
print('\n잔차 기준 검정 결과 요약')
print(detrended_test_summary.to_string(index=False))
detrended_test_summary

잔차 기준 Kruskal-Wallis 검정
귀무가설: 트렌드 제거 후 계절별 차이 없다
경부선: p-value=0.000003 → 트렌드 제거 후 계절별 유의미한 차이 있음
호남선: p-value=0.000000 → 트렌드 제거 후 계절별 유의미한 차이 있음
경전선: p-value=0.000040 → 트렌드 제거 후 계절별 유의미한 차이 있음
전라선: p-value=0.011698 → 트렌드 제거 후 계절별 유의미한 차이 있음
동해선: p-value=0.042544 → 트렌드 제거 후 계절별 유의미한 차이 있음

잔차 기준 검정 결과 요약
 노선                검정방법  p-value                      해석
경부선 Kruskal-Wallis (잔차) 0.000003 트렌드 제거 후 계절별 유의미한 차이 있음
호남선 Kruskal-Wallis (잔차) 0.000000 트렌드 제거 후 계절별 유의미한 차이 있음
경전선 Kruskal-Wallis (잔차) 0.000040 트렌드 제거 후 계절별 유의미한 차이 있음
전라선 Kruskal-Wallis (잔차) 0.011698 트렌드 제거 후 계절별 유의미한 차이 있음
동해선 Kruskal-Wallis (잔차) 0.042544 트렌드 제거 후 계절별 유의미한 차이 있음


,노선,검정방법,p-value,해석
0,경부선,Kruskal-Wallis (잔차),0.000003,트렌드 제거 후 계절별 유의미한 차이 있음
1,호남선,Kruskal-Wallis (잔차),0.000000,트렌드 제거 후 계절별 유의미한 차이 있음
2,경전선,Kruskal-Wallis (잔차),0.000040,트렌드 제거 후 계절별 유의미한 차이 있음
3,전라선,Kruskal-Wallis (잔차),0.011698,트렌드 제거 후 계절별 유의미한 차이 있음
4,동해선,Kruskal-Wallis (잔차),0.042544,트렌드 제거 후 계절별 유의미한 차이 있음


### 계절별 평균 잔차

In [11]:
seasonal_residual_mean = (
    df_detrended.pivot_table(
        index='노선',
        columns='계절',
        values='공실률_잔차',
        aggfunc='mean',
        observed=True,
    )
    .reindex(index=ROUTE_ORDER, columns=SEASON_ORDER)
    .round(2)
)

print('노선별 × 계절별 평균 잔차')
print('양수는 해당 노선·연도 평균보다 공실률이 높고, 음수는 평균보다 낮다는 뜻이다.')
print(seasonal_residual_mean.to_string())
seasonal_residual_mean

print('\n계절별 평균 잔차 상세 해석')
for route in ROUTE_ORDER:
    if route not in seasonal_residual_mean.index:
        continue

    route_residuals = seasonal_residual_mean.loc[route].dropna()
    if route_residuals.empty:
        print(f'{route}: 해석 가능한 계절별 잔차가 없음')
        continue

    high_season = route_residuals.idxmax()
    low_season = route_residuals.idxmin()
    high_value = route_residuals.max()
    low_value = route_residuals.min()
    print(
        f'{route}: {high_season}은 연도 평균보다 공실률이 {high_value:+.2f}%p 높고, '
        f'{low_season}은 {low_value:+.2f}%p 낮음'
    )

노선별 × 계절별 평균 잔차
양수는 해당 노선·연도 평균보다 공실률이 높고, 음수는 평균보다 낮다는 뜻이다.
계절      봄    여름    가을    겨울
노선                         
경부선  2.23  1.33 -3.40 -0.13
호남선  1.02  2.61 -4.20  0.59
경전선  1.99  2.43 -4.14 -0.28
전라선  2.15  0.97 -3.51  0.55
동해선  2.90  0.10 -4.13  1.32

계절별 평균 잔차 상세 해석
경부선: 봄은 연도 평균보다 공실률이 +2.23%p 높고, 가을은 -3.40%p 낮음
호남선: 여름은 연도 평균보다 공실률이 +2.61%p 높고, 가을은 -4.20%p 낮음
경전선: 여름은 연도 평균보다 공실률이 +2.43%p 높고, 가을은 -4.14%p 낮음
전라선: 봄은 연도 평균보다 공실률이 +2.15%p 높고, 가을은 -3.51%p 낮음
동해선: 봄은 연도 평균보다 공실률이 +2.90%p 높고, 가을은 -4.13%p 낮음


### 결과 해석

연도별 노선 평균을 제거한 뒤에도 대부분 노선의 계절별 차이는 통계적으로 유의하지 않다. 따라서 현재 월 단위 데이터에서는 계절 자체보다 노선별 수요 구조, 개통 시점, 코로나 기간, SRT 개통 이후 변화 같은 요인이 공실률 차이를 더 크게 설명할 가능성이 있다.

계절별 평균 잔차는 같은 노선·같은 연도 평균 대비 어느 계절에 공실률이 상대적으로 높거나 낮은지 보여준다. 양수 계절은 평균보다 빈 좌석이 많은 시기이고, 음수 계절은 평균보다 좌석 여유가 적거나 초과수요가 강한 시기로 해석한다. 단, 이는 월 단위 집계 데이터 기반의 탐색 결과이므로 요일·명절·운행횟수·좌석 공급량 변수를 추가한 뒤 재검토하는 것이 바람직하다.